<center><h1>Deng_Kaiyue_HW3</h1></center>
<br>
<br>

Name: Kaiyue Deng
<br>
Github Username: kai-dky
<br>
USC ID: 7459633091

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

Get the AReM Data Set

In [2]:
data_root = Path("../data/AReM")

activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

column_names = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

time_series_columns = column_names[1:]
expected_time = pd.Index(np.arange(0, 120000, 250), name="time")


def load_instance(file_path):
    instance_data = pd.read_csv(
        file_path,
        skiprows=5,
        header=None,
        names=column_names,
        usecols=range(7),
        sep=r"[,\s]+",
        engine="python"
    )

    instance_data = instance_data.apply(
        pd.to_numeric,
        errors="raise"
    )

    instance_data = (
        instance_data
        .set_index("time")
        .reindex(expected_time)
    )

    instance_data[time_series_columns] = (
        instance_data[time_series_columns]
        .interpolate(method="linear", limit_direction="both")
    )

    return instance_data.reset_index()


instances = []

for activity in activities:
    activity_folder = data_root / activity

    dataset_files = sorted(
        activity_folder.glob("dataset*.csv"),
        key=lambda path: int(path.stem.replace("dataset", ""))
    )

    for file_path in dataset_files:
        dataset_number = int(
            file_path.stem.replace("dataset", "")
        )

        instances.append({
            "activity": activity,
            "dataset": dataset_number,
            "data": load_instance(file_path)
        })


print(f"Number of activities: {len(activities)}")
print(f"Number of instances: {len(instances)}")
print(
    "All instances contain 480 observations:",
    all(len(instance["data"]) == 480 for instance in instances)
)

display(instances[0]["data"].head())

Number of activities: 7
Number of instances: 88
All instances contain 480 observations: True


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


### (b) Test and Train Data

In [3]:
train_instances = []
test_instances = []

for instance in instances:
    activity = instance["activity"]
    dataset_number = instance["dataset"]

    if activity in {"bending1", "bending2"}:
        is_test = dataset_number in {1, 2}
    else:
        is_test = dataset_number in {1, 2, 3}

    instance["split"] = "test" if is_test else "train"

    if is_test:
        test_instances.append(instance)
    else:
        train_instances.append(instance)


split_information = pd.DataFrame([
    {
        "Activity": instance["activity"],
        "Dataset": instance["dataset"],
        "Split": instance["split"]
    }
    for instance in instances
])

split_summary = pd.crosstab(
    split_information["Activity"],
    split_information["Split"]
)

display(split_summary)

print(f"Number of training instances: {len(train_instances)}")
print(f"Number of test instances: {len(test_instances)}")

Split,test,train
Activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


Number of training instances: 69
Number of test instances: 19


### (c) Feature Extraction

#### i. Research

Commonly used time-domain features for time-series classification include:

- Location and magnitude features: minimum, maximum, mean, median, and quantiles.
- Dispersion features: range, variance, standard deviation, interquartile range, and median absolute deviation.
- Distribution-shape features: skewness and kurtosis.
- Energy features: root mean square and absolute energy.
- Change and trend features: mean absolute change, sum of absolute changes, slope, and first differences.
- Pattern features: zero-crossing rate, number of peaks, and the lengths of runs above or below the mean.
- Dependence features: autocorrelation and partial autocorrelation.

For this assignment, the required features are minimum, maximum, mean, median, standard deviation, first quartile, and third quartile.

**References**

1. [tsfresh Feature Calculators Documentation](https://tsfresh.readthedocs.io/en/latest/api/tsfresh.html)
2. [Christ et al., Time Series FeatuRe Extraction on basis of Scalable Hypothesis Tests](https://www.sciencedirect.com/science/article/pii/S0925231218304843)

#### ii. Extraction

In [4]:
def extract_features(instance_data):
    extracted_features = {}

    for series_number, column in enumerate(
        time_series_columns,
        start=1
    ):
        series = instance_data[column]

        extracted_features.update({
            f"min{series_number}": series.min(),
            f"max{series_number}": series.max(),
            f"mean{series_number}": series.mean(),
            f"median{series_number}": series.median(),
            f"std{series_number}": series.std(ddof=1),
            f"q1_{series_number}": series.quantile(0.25),
            f"q3_{series_number}": series.quantile(0.75)
        })

    return extracted_features


feature_rows = []

for instance_number, instance in enumerate(instances, start=1):
    feature_row = {
        "Instance": instance_number,
        "Activity": instance["activity"],
        "Dataset": instance["dataset"],
        "Split": instance["split"]
    }

    feature_row.update(
        extract_features(instance["data"])
    )

    feature_rows.append(feature_row)


feature_data = pd.DataFrame(feature_rows)

metadata_columns = [
    "Instance",
    "Activity",
    "Dataset",
    "Split"
]

feature_columns = [
    column
    for column in feature_data.columns
    if column not in metadata_columns
]

print(f"Number of instances: {feature_data.shape[0]}")
print(f"Number of extracted features: {len(feature_columns)}")
print(f"Complete dataset shape: {feature_data.shape}")

display(feature_data.head().round(4))

Number of instances: 88
Number of extracted features: 42
Complete dataset shape: (88, 46)


,Instance,Activity,Dataset,Split,min1,max1,mean1,median1,std1,q1_1,...,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
0,1,bending1,1,test,37.25,45.00,40.6248,40.50,1.4770,39.25,...,2.1884,33.0000,36.00,0.0,1.92,0.5706,0.43,0.5829,0.0,1.30
1,2,bending1,2,test,38.00,45.67,42.8128,42.50,1.4355,42.00,...,1.9953,32.0000,34.50,0.0,3.11,0.5711,0.43,0.6010,0.0,1.30
2,3,bending1,3,train,35.00,47.40,43.9545,44.33,1.5588,43.00,...,1.9996,35.3625,36.50,0.0,1.79,0.4933,0.43,0.5135,0.0,0.94
3,4,bending1,4,train,33.00,47.75,42.1798,43.50,3.6707,39.15,...,3.8494,30.4575,36.33,0.0,2.18,0.6135,0.50,0.5243,0.0,1.00
4,5,bending1,5,train,33.00,45.75,41.6781,41.75,2.2435,41.33,...,2.4110,28.4575,31.25,0.0,1.79,0.3833,0.43,0.3892,0.0,0.50


#### iii. Standard Deviation

In [5]:
random_generator = np.random.default_rng(42)

number_of_bootstrap_samples = 10000
bootstrap_results = []

for feature in feature_columns:
    feature_values = feature_data[feature].to_numpy()

    estimated_standard_deviation = feature_values.std(ddof=1)

    bootstrap_indices = random_generator.integers(
        low=0,
        high=len(feature_values),
        size=(
            number_of_bootstrap_samples,
            len(feature_values)
        )
    )

    bootstrap_samples = feature_values[bootstrap_indices]

    bootstrap_standard_deviations = bootstrap_samples.std(
        axis=1,
        ddof=1
    )

    confidence_interval_lower = np.percentile(
        bootstrap_standard_deviations,
        5
    )

    confidence_interval_upper = np.percentile(
        bootstrap_standard_deviations,
        95
    )

    bootstrap_results.append({
        "Feature": feature,
        "Standard Deviation": estimated_standard_deviation,
        "90% CI Lower": confidence_interval_lower,
        "90% CI Upper": confidence_interval_upper
    })


bootstrap_results_df = pd.DataFrame(
    bootstrap_results
).set_index("Feature")

display(bootstrap_results_df.round(4))

,Standard Deviation,90% CI Lower,90% CI Upper
Feature,,,
min1,9.5700,8.2360,10.7739
max1,4.3944,3.3299,5.2966
mean1,5.3357,4.6883,5.8891
median1,5.4401,4.7958,5.9873
std1,1.7722,1.5742,1.9517
q1_1,6.1536,5.5759,6.6435
q3_1,5.1389,4.3263,5.8427
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6311,5.3999


#### iv. Select Features

I select minimum, mean and maximum as the three most important time-domain features.

The minimum and maximum describe the lower and upper extremes of each time series, while the mean describes its overall central level. Together, these three features provide a simple and interpretable summary of the range and typical magnitude of the sensor measurements. They may therefore help distinguish activities that produce different sensor levels and ranges.

## 2. ISLR 3.7.4

### (a) Linear Train

The training RSS for the cubic regression will be less than or equal to the training RSS for the linear regression. The linear model is a special case of the cubic model obtained by setting the coefficients of \(X^2\) and \(X^3\) equal to zero. Because the cubic model has two additional predictors, least squares can always reproduce the linear fit and may also fit some of the random noise in the training data. Therefore, although the true relationship is linear, the cubic regression cannot have a higher training RSS and will typically have a slightly lower training RSS.

### (b) Linear Test

The linear regression is expected to have a lower test RSS than the cubic regression. Because the true relationship between \(X\) and \(Y\) is linear, the additional \(X^2\) and \(X^3\) terms are unnecessary. These extra terms make the cubic model more flexible and increase its variance, so it may fit random noise in the training data and perform worse on unseen test data. Therefore, although this is not guaranteed for every particular test set, the linear regression should have a lower test RSS on average.

### (c) Not Linear Train

The cubic regression will have a lower than or equal training RSS compared with the linear regression. This conclusion does not depend on the true relationship between \(X\) and \(Y\). The linear model is nested within the cubic model because the cubic model can reproduce the linear model by setting the coefficients of \(X^2\) and \(X^3\) equal to zero. Therefore, the additional flexibility of the cubic regression allows it to fit the training data at least as well as the linear regression.

### (d) Not Linear Testing

There is not enough information to determine which model will have the lower test RSS. If the true nonlinear relationship can be approximated well by a cubic function, the cubic regression may have a lower test RSS because it can capture the curvature that the linear model misses. However, if the true relationship is only slightly nonlinear or has a form that is not well represented by a cubic function, the cubic model’s additional flexibility may lead to overfitting and a higher test RSS. Therefore, either model could perform better on the test data.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 